# 1. Instalar Dependencias



In [1]:
%%capture
# Instalar librerías necesarias
!pip install -q transformers==4.31.0
!pip install -q peft==0.4.0
!pip install -q accelerate==0.21.0
!pip install -q datasets==2.14.4
!pip install -q bitsandbytes==0.41.0
!pip install -q sentencepiece==0.1.99

print("✅ Dependencias instaladas")

# 2. Verificar GPU

In [1]:
import torch

print("=" * 70)
print("🎮 VERIFICACIÓN DE GPU")
print("=" * 70)

if torch.cuda.is_available():
    print(f"✅ GPU disponible: {torch.cuda.get_device_name(0)}")
    print(f"📊 Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"🚀 CUDA Version: {torch.version.cuda}")
else:
    print("❌ GPU NO disponible")
    print("⚠️  Ve a Runtime → Change runtime type → T4 GPU")

print("=" * 70)

🎮 VERIFICACIÓN DE GPU
✅ GPU disponible: Tesla T4
📊 Memoria GPU: 15.83 GB
🚀 CUDA Version: 12.6


# 3. Cargar Dataset de entrenamiento

In [5]:
from google.colab import files
import json

print("📤 Sube tu archivo dataset_pedagogico.json")
print("   (Haz clic en 'Choose Files' y selecciona el archivo)")
print()

uploaded = files.upload()

# Verificar que se subió correctamente
if 'dataset_pedagogico.json' in uploaded:
    with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"\n✅ Dataset cargado: {len(data)} ejemplos")
    print(f"\n📝 Primer ejemplo:")
    print(f"   Instrucción: {data[0]['instruction'][:50]}...")
    print(f"   Input: {data[0]['input'][:50]}...")
else:
    print("❌ Error: No se encontró dataset_pedagogico.json")

📤 Sube tu archivo dataset_pedagogico.json
   (Haz clic en 'Choose Files' y selecciona el archivo)



Saving dataset_pedagogico.json to dataset_pedagogico.json

✅ Dataset cargado: 30 ejemplos

📝 Primer ejemplo:
   Instrucción: Explica qué es una derivada...
   Input: Necesito entender el concepto de derivada para mi ...


# 4. Abrir TensorBoard (⚠️ EJECUTAR ANTES DE ENTRENAR)

**IMPORTANTE:** Ejecuta esta celda ANTES de entrenar para ver las curvas de loss en tiempo real.

Verás 2 curvas comparativas:
- **logs/tinyllama** (azul) - TinyLlama
- **logs/phi2** (naranja) - Phi-2

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

print("\n📊 TensorBoard abierto")
print("   Se actualizará cada 30 segundos")
print("   Verás 2 curvas comparativas:")
print("   - logs/tinyllama (azul)")
print("   - logs/phi2 (naranja)")
print("\n⚠️  IMPORTANTE: Deja esta celda ejecutando mientras entrenas")

# 5. Entrenamiento con TinyLlama (10-15 min)

**Tiempo estimado:** 10-15 minutos
**Curva:** Azul en TensorBoard

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

print("=" * 70)
print("🚀 ENTRENAMIENTO TINYLLAMA - CURVA AZUL EN TENSORBOARD")
print("=" * 70)

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

LORA_CONFIG = {
    "r": 8,
    "lora_alpha": 16,
    "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
    "lora_dropout": 0.1,
    "bias": "none",
    "task_type": TaskType.CAUSAL_LM
}

TRAINING_CONFIG = {
    "output_dir": "./lora_tinyllama",
    "num_train_epochs": 20,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 5e-5,
    "fp16": True,
    "logging_dir": "./logs/tinyllama",  # ← TENSORBOARD: Curva azul
    "logging_steps": 5,
    "save_steps": 100,
    "save_total_limit": 2,
    "warmup_steps": 20,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "report_to": "tensorboard",  # ← Activado
    "disable_tqdm": False,
    "logging_first_step": True,
}

print(f"\n📦 Modelo: {MODEL_NAME}")
print(f"⏱️  Tiempo: 10-15 minutos")
print(f"📊 TensorBoard: logs/tinyllama (azul)")

# Preparar dataset
print("\n📚 Preparando dataset...")
with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def format_instruction(example):
    text = f"""### Instrucción:
{example['instruction']}

IMPORTANTE: Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.

### Entrada:
{example['input']}

### Respuesta:
{example['output']}"""
    return {"text": text}

dataset = Dataset.from_list(data).map(format_instruction)
print(f"   ✅ {len(dataset)} ejemplos")

# Cargar modelo
print("\n🤖 Cargando modelo...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print("   ✅ Modelo cargado")

# Aplicar LoRA
print("\n🔧 Aplicando LoRA...")
lora_config = LoraConfig(**LORA_CONFIG)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n📊 Entrenables: {trainable:,} ({100*trainable/total:.4f}%)")

# Tokenizar
print("\n📝 Tokenizando...")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("   ✅ Dataset tokenizado")

# Entrenar
print("\n" + "=" * 70)
print("🚀 INICIANDO ENTRENAMIENTO")
print("=" * 70)
print("⏱️  10-15 minutos")
print("📊 Mira TensorBoard arriba ↑ (curva azul)")
print()

training_args = TrainingArguments(**TRAINING_CONFIG)
trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_dataset, data_collator=data_collator)

trainer.train()

print("\n✅ COMPLETADO")
final_loss = trainer.state.log_history[-1].get('loss', 'N/A')
print(f"📊 Loss final: {final_loss}")

# Guardar
print("\n💾 Guardando adaptadores...")
model.save_pretrained("./lora_adapters_tinyllama")
tokenizer.save_pretrained("./lora_adapters_tinyllama")
print("   ✅ Guardado en: ./lora_adapters_tinyllama")

🚀 ENTRENAMIENTO CON PEFT (LoRA) + GPU - CONFIGURACIÓN OPTIMIZADA V2

📦 Modelo base: TinyLlama/TinyLlama-1.1B-Chat-v1.0
🔧 LoRA config: r=8, alpha=16
🏋️  Training: 20 épocas (AUMENTADO)
📊 Batch efectivo: 8
📉 Learning rate: 5e-05
⏱️  Tiempo estimado: 40-50 minutos

📚 Preparando dataset...


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

   ✅ 30 ejemplos preparados

🤖 Cargando modelo y tokenizer...
   ✅ Modelo cargado en GPU

🔧 Aplicando adaptadores LoRA...

📊 ESTADÍSTICAS:
   Total: 1,102,301,184 parámetros
   Entrenables: 2,252,800 (0.2044%)

📝 Tokenizando dataset...


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


   ✅ Dataset tokenizado

🚀 INICIANDO ENTRENAMIENTO CON GPU - 20 ÉPOCAS
⏱️  Tiempo estimado: 40-50 minutos
📉 Loss esperado: Debe bajar de 1.5 a <0.5
🎯 Objetivo: Loss final <0.5 para respuestas coherentes



Step,Training Loss
5,1.555300
10,1.567200
15,1.563000
20,1.516400
25,1.490300
30,1.442700
35,1.362600
40,1.353100
45,1.321900
50,1.275100



✅ ENTRENAMIENTO COMPLETADO

📊 Loss final: N/A

💾 Guardando adaptadores LoRA...
   ✅ Adaptadores guardados en: ./lora_adapters

🧪 PROBANDO MODELO...
📝 RESPUESTA DEL MODELO:
----------------------------------------------------------------------
### Instrucción:
Explica qué es una derivada

IMPORTANTE: Responde SIEMPRE en español con tono pedagógico, motivador y amigable. Usa emojis apropiados.

### Entrada:
Necesito entender el concepto de derivada

### Respuesta:
¡Hola! ¡Es importante saber cómo se calcula la derivada! La derivada es un número que permite calcular las funciones sucesivas o diferentes. A continuación, te explicaré por qué es importante conocerla:

**Ejemplo 1:** Si deseas calcular la derivada del fijo `y = x^2` (en función del valor de `x`), puedes usar la siguiente forma:

```
f'(x) = 2 * x
```

En este caso, `f(x)` sería `x^2`. Entonces:

```
f'(x) = 2 * x
f(x) - f(0) / Δx = 2 * (x + 0)
```

Para obtener la derivada actualmente, solo tienes que escribirlo en la forma 

# 6. Entrenamiento con Phi-2 (15-20 min)

**Tiempo estimado:** 15-20 minutos
**Curva:** Naranja en TensorBoard

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

print("=" * 70)
print("🚀 ENTRENAMIENTO PHI-2 - CURVA NARANJA EN TENSORBOARD")
print("=" * 70)

MODEL_NAME = "microsoft/phi-2"

LORA_CONFIG = {
    "r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "v_proj", "k_proj", "dense"],
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": TaskType.CAUSAL_LM
}

TRAINING_CONFIG = {
    "output_dir": "./lora_phi2",
    "num_train_epochs": 15,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "fp16": True,
    "logging_dir": "./logs/phi2",  # ← TENSORBOARD: Curva naranja
    "logging_steps": 5,
    "save_steps": 100,
    "save_total_limit": 2,
    "warmup_steps": 30,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "report_to": "tensorboard",  # ← Activado
    "disable_tqdm": False,
    "logging_first_step": True,
}

print(f"\n📦 Modelo: {MODEL_NAME}")
print(f"⏱️  Tiempo: 15-20 minutos")
print(f"📊 TensorBoard: logs/phi2 (naranja)")
print(f"💡 Phi-2 es mejor que TinyLlama en español")

# Preparar dataset
print("\n📚 Preparando dataset...")
with open('dataset_pedagogico.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def format_instruction(example):
    text = f"""Instrucción: {example['instruction']}

Contexto: {example['input']}

Respuesta (en español, tono pedagógico y motivador):
{example['output']}"""
    return {"text": text}

dataset = Dataset.from_list(data).map(format_instruction)
print(f"   ✅ {len(dataset)} ejemplos")

# Cargar modelo
print("\n🤖 Cargando Phi-2...")
print("   (Descargando ~5GB, 2-3 minutos)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, add_eos_token=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
print("   ✅ Phi-2 cargado")

# Aplicar LoRA
print("\n🔧 Aplicando LoRA...")
lora_config = LoraConfig(**LORA_CONFIG)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n📊 Entrenables: {trainable:,} ({100*trainable/total:.4f}%)")

# Tokenizar
print("\n📝 Tokenizando...")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print("   ✅ Dataset tokenizado")

# Entrenar
print("\n" + "=" * 70)
print("🚀 INICIANDO ENTRENAMIENTO PHI-2")
print("=" * 70)
print("⏱️  15-20 minutos")
print("📊 Mira TensorBoard arriba ↑ (curva naranja)")
print()

training_args = TrainingArguments(**TRAINING_CONFIG)
trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_dataset, data_collator=data_collator)

trainer.train()

print("\n✅ COMPLETADO")
final_loss = trainer.state.log_history[-1].get('loss', 'N/A')
print(f"📊 Loss final: {final_loss}")

if isinstance(final_loss, float):
    if final_loss < 0.3:
        print("✅ ¡Excelente! Loss <0.3")
    elif final_loss < 0.5:
        print("✅ Muy bien! Loss <0.5")

# Guardar
print("\n💾 Guardando adaptadores...")
model.save_pretrained("./lora_adapters_phi2")
tokenizer.save_pretrained("./lora_adapters_phi2")
print("   ✅ Guardado en: ./lora_adapters_phi2")

🚀 ENTRENAMIENTO CON PHI-2 (MICROSOFT) - MEJOR MODELO

📦 Modelo base: microsoft/phi-2
🔧 LoRA config: r=16, alpha=32
🏋️  Training: 15 épocas
📊 Batch efectivo: 8
📉 Learning rate: 0.0002
⏱️  Tiempo estimado: 60 minutos

💡 Phi-2 es 3-4x mejor que TinyLlama en español

📚 Preparando dataset...


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

   ✅ 30 ejemplos preparados

🤖 Cargando Phi-2 y tokenizer...
   (Esto puede tardar 2-3 minutos, descargando ~5GB)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

   ✅ Dataset tokenizado

🚀 INICIANDO ENTRENAMIENTO CON PHI-2
⏱️  Tiempo estimado: 60 minutos
📉 Loss esperado: Debe bajar de 1.5 a <0.3
🎯 Phi-2 aprende más rápido y mejor que TinyLlama



Step,Training Loss
5,1.548800
10,1.569900
15,1.505200
20,1.382400
25,1.292100
30,1.166300
35,1.025400
40,0.993200
45,0.925600
50,0.872200



✅ ENTRENAMIENTO COMPLETADO

📊 Loss final: N/A

💾 Guardando adaptadores LoRA de Phi-2...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


   ✅ Adaptadores guardados en: ./lora_adapters

🧪 PROBANDO PHI-2 ENTRENADO...
📝 RESPUESTA DE PHI-2:
----------------------------------------------------------------------
Instrucción: Explica qué es una derivada

Contexto: Necesito entender el concepto de derivada

Respuesta (en español, tono pedagógico y motivador):
¡Excelente pregunta! La derivada es fundamental en matemáticas. 🧮

**Definitiva:**
La derivada del ejercicio x² = 4 dx es la regla que calcule cambios de x respecto a dólares.

```python
import sympy as sp
x = sp.Symbol('x')
derivative_dx2 = sp.diff(4*x)  # Calcular la derivada
print(f"Derivado de {4}x²: {derivative_dx2}")
```

**Fase 1: Interpretar con posibles fases:**
- Cualquier funcional tiene at least una fase normal
- Normalizamos las fases para compararlles

**Fase 2: Identificar profundidades:**
- Una fase local maximiza o minimaza
- Puede encontrar bordes del índice de dépensión
- Es común usar secantes por filtro

**Practical Ejemplos:**
1. Calculando velocidad 

# 7. Comparar Resultados en TensorBoard

Mira TensorBoard arriba para ver las 2 curvas comparadas:
- **Azul:** TinyLlama
- **Naranja:** Phi-2

Phi-2 debería tener loss más bajo (mejor).

# 8. Descargar Adaptadores

Elige cuál modelo te funcionó mejor según TensorBoard.

In [ ]:
import shutil
from google.colab import files

print("📦 Elige qué modelo descargar:")
print("   1. TinyLlama (./lora_adapters_tinyllama)")
print("   2. Phi-2 (./lora_adapters_phi2)")
print()

# Descomentar el que quieras descargar:

# OPCIÓN 1: TinyLlama
# shutil.make_archive('lora_adapters_tinyllama', 'zip', './lora_adapters_tinyllama')
# files.download('lora_adapters_tinyllama.zip')
# print("✅ TinyLlama descargado")
# print("Actualiza .env: HUGGINGFACE_MODEL=TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# OPCIÓN 2: Phi-2 (RECOMENDADO)
shutil.make_archive('lora_adapters_phi2', 'zip', './lora_adapters_phi2')
files.download('lora_adapters_phi2.zip')
print("✅ Phi-2 descargado")
print("Actualiza .env: HUGGINGFACE_MODEL=microsoft/phi-2")